# **Renge Göre Basit Nesne Takibi**

####** Bu derste şunları öğreneceğiz:**
1. Bir Maske Oluşturmak ve ardından İstediğimiz Nesneyi İzlemek için HSV Renk Filtresi nasıl kullanılır


In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt

def imshow(title = "Image", image = None, size = 10):
    w, h = image.shape[0], image.shape[1]
    aspect_ratio = w/h
    plt.figure(figsize=(size * aspect_ratio,size))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.show()

In [ ]:
#Örnek Videomuzu izleyelim
from IPython.display import Video, display
display(Video("multi_shape_tracking_example.mp4", embed=True))

In [ ]:
# Obje Takibi (Genel)
import cv2
import numpy as np

# Nesnelerin merkez noktalarını saklamak için liste (birden fazla nesne için)
points = []

# Video akışını yükleyelim
cap = cv2.VideoCapture('multi_shape_tracking_example.mp4')

# Videonun genişlik ve yüksekliğini integer olarak alalım
width = int(cap.get(3)) 
height = int(cap.get(4))

# HSV (Hue, Saturation, Value) renk uzayında yeşil renge yakın bir aralık belirlenmiş.
# Sadece bu renkler algılanacak. (Burayı istediğin renge göre değiştirebilirsin)
lower = np.array([50, 100, 100])
upper = np.array([70, 255, 255])

# Codec'i tanımlayın ve VideoWriter nesnesini oluşturun. Çıktı '*.avi' dosyasında saklanır.
out = cv2.VideoWriter('object_tracking_output.avi', 
                      cv2.VideoWriter_fourcc('M','J','P','G'), 
                      30, (width, height))

ret, frame = cap.read()
Height, Width = frame.shape[:2]
frame_count = 0

while True:
    # Çerçeveyi yakala
    ret, frame = cap.read()
    if not ret:
        break

    # HSV uzayına dönüştür
    hsv_img = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # Yalnızca belirlenen renk aralığını maskele
    mask = cv2.inRange(hsv_img, lower, upper)

    # Maskedeki şekilleri bulur
    contours, _ = cv2.findContours(mask.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # Birden fazla nesne için her konturu dolaş
    for c in contours:
        (x, y), radius = cv2.minEnclosingCircle(c)
        M = cv2.moments(c)

        if M["m00"] != 0:
            center = (int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"]))
        else:
            center = (int(Width/2), int(Height/2))

        # Çok küçük gürültüleri engellemek için minimum yarıçap kontrolü
        if radius > 15:  
            # Her nesneye çember ve merkez noktası çiz
            cv2.circle(frame, (int(x), int(y)), int(radius), (0, 0, 255), 2)
            cv2.circle(frame, center, 5, (0, 255, 0), -1)

            # Noktayı kaydet (tüm nesneler için)
            points.append(center)

    # Önceki noktalardan iz çiz (tüm nesneler için ortak iz)
    for i in range(1, len(points)):
        try:
            cv2.line(frame, points[i - 1], points[i], (0, 255, 0), 2)
        except:
            pass

    out.write(frame)

cap.release()
out.release()
print("Tamamlandı")

In [ ]:
#Kaydedilen videoyu oynatmak için önce .mp4 e çevirelim. Ardından oynatalım
cap = cv2.VideoCapture('object_tracking_output.avi')
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter('object_tracking_output.mp4', cv2.VideoWriter_fourcc(*'mp4v'), 30, (w, h))

while True:
    ret, frame = cap.read()
    if not ret:
        break
    out.write(frame)

cap.release()
out.release()

from IPython.display import Video, display
display(Video("object_tracking_output.mp4", embed=True))